In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_tavily import TavilySearch

import psycopg2
from psycopg2 import Error

In [3]:
load_dotenv()

True

In [ ]:
GROQ_API_KEY = os.getenv("GROK_API_KEY")
llm = ChatGroq(model = "qwen/qwen3.6-27b",api_key=GROQ_API_KEY)

# LANGCHAIN TAVILY SEARCH TOOL

In [31]:
@tool
def tavily_search_tool(query : str):
    """This tool shall be used when the information about the topic needs to be extracted from the internet"""
    search = TavilySearch(
        max_results = 3,
        search_depth = "basic",
        include_answer = True
    )
    return search.invoke({"query":query})

In [36]:
tools = [tavily_search_tool]

Tavily = llm.bind_tools(tools)

Tavily.invoke("who is the current Prime minister of united Kingdom ?").tool_calls

[{'name': 'tavily_search_tool',
  'args': {'query': 'current Prime Minister of the United Kingdom 2026'},
  'id': 'ybz0gam71',
  'type': 'tool_call'}]

# SQL QUERY TOOL

In [6]:
@tool
def pgsql_query_tool(query : str):
    """Use this tool to query the postgresql database about the company's performance"""
    db_config = {
        "db_name" : "company_performance",
        "user" : "user",
        "password" : "password",
        "host" : "127.0.0.1",
        "port" : "5432"
    }
    try:
        with psycopg2.connect(**db_config) as conn:
            with conn.cursor as cur:

                cur.execute(query)
                print("query executed successfully")
                rows = cur.fetchall()
                return rows

    except Exception as error:
        print(f"Database error occured {error}")

In [7]:
tools =[pgsql_query_tool]

llm_with_pgsql = llm.bind_tools(tools)

llm_with_pgsql.invoke("How did Apple perform compared to Microsoft in Q3").tool_calls

[{'name': 'pgsql_query_tool',
  'args': {'query': "SELECT * FROM company_performance WHERE company_name IN ('Apple', 'Microsoft') AND quarter = 'Q3' ORDER BY year DESC LIMIT 10;"},
  'id': '1n34ysty7',
  'type': 'tool_call'}]